# KNN — PCA 이후 투표 재검증

PCA 선행 조건을 같은 Notebook에서 재구성. 전체 26개 조합.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("KNN", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
knn=model; tr=knn._fit_X; ty=knn._y; kbase=base
print("Local output:",root)

In [ ]:
def vote(d,ids,k,alpha,train_y=ty):
    ds=d[:,:k]; ls=train_y[ids[:,:k]]
    zero=ds==0
    weights=np.power(np.maximum(ds,1e-30),-alpha)
    weights=np.where(zero.any(axis=1)[:,None],zero,weights)
    scores=np.zeros((len(ds),16))
    np.add.at(scores,(np.arange(len(ds))[:,None],ls),weights)
    return np.argmax(scores,axis=1)


In [ ]:
from sklearn.decomposition import PCA
t=time.perf_counter()
pc=PCA(n_components=512,svd_solver='covariance_eigh',whiten=False)
pc.fit(tr)
pca_fit_sec=time.perf_counter()-t
print('PCA fit',pca_fit_sec,pc.explained_variance_ratio_.sum())
joblib.dump(pc,root/'03_KNN/pca512_training_only.joblib',compress=3)
np.savez(root/'03_KNN/pca_variance.npz',ratios=pc.explained_variance_ratio_)

In [ ]:
# PCA full training projection; nested prefixes share exact fitted eigenspace.
t=time.perf_counter()
zt=pc.transform(tr);zv=pc.transform(Xv);zo=pc.transform(Xo)
pca_transform_sec=time.perf_counter()-t
print(zt.shape,zt.dtype,pca_transform_sec)

In [ ]:
pca_rows=[{'condition':'PCA00','dims':1108,'explained_var':1.0,'scope':'full_training_full_handmade',**stats(kbase,kbase)}]
pca_preds={'PCA00':kbase}
for dd in [512,256,128,64]:
    ztrain=np.ascontiguousarray(zt[:,:dd]);zzv=np.ascontiguousarray(zv[:,:dd])
    t=time.perf_counter()
    mod=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(ztrain,ty)
    ft=time.perf_counter()-t;t=time.perf_counter();pred=mod.predict(zzv);et=time.perf_counter()-t
    rec={'condition':f'PCA{dd}','dims':dd,'explained_var':float(pc.explained_variance_ratio_[:dd].sum()),'scope':'full_training_full_handmade','pca_fit_s_shared':pca_fit_sec,'pca_transform_512_s_shared':pca_transform_sec,'knn_fit_s':ft,'batch_eval_s':et,**stats(pred,kbase)}
    pca_rows.append(rec);pca_preds[f'PCA{dd}']=pred
    print(rec['condition'],rec['correct'],rec['macro_f1'],rec['explained_var'],et,flush=True)
    del ztrain,zzv,mod
    gc.collect()
pd.DataFrame(pca_rows).to_csv(root/'03_KNN/EXP-FT-KNN-004_handmade.csv',index=False)
pd.DataFrame({'true':y,**pca_preds}).to_csv(root/'03_KNN/EXP-FT-KNN-004_predictions.csv',index=False)

In [ ]:
# Recheck vote settings after PCA for the two equal-accuracy speed candidates.
pvrows=[];pvp={}
for dd in [128,512]:
 zztr=np.ascontiguousarray(zt[:,:dd]);zzv=np.ascontiguousarray(zv[:,:dd])
 mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(zztr,ty)
 dp,ip=mm.kneighbors(zzv,n_neighbors=9)
 assert np.array_equal(vote(dp,ip,3,1),pca_preds[f'PCA{dd}'])
 for k in [1,3,5,7,9]:
  for al in ([1] if k==1 else [.5,1,2]):
   pr=vote(dp,ip,k,al)
   tag=f'PCA{dd}_k{k}_a{al}'
   pvrows.append({'condition':tag,'dims':dd,'k':k,'alpha':al,**stats(pr,kbase)});pvp[tag]=pr
 del zztr,zzv,mm
 gc.collect()
pd.DataFrame(pvrows).to_csv(root/'03_KNN/EXP-FT-KNN-005_pca_vote_recheck.csv',index=False)
pd.DataFrame({'true':y,**pvp}).to_csv(root/'03_KNN/EXP-FT-KNN-005_predictions.csv',index=False)
print(pd.DataFrame(pvrows).sort_values(['correct','macro_f1'],ascending=False)[['condition','correct','macro_precision','macro_f1']].head(12).to_string(index=False))